# Stage 2 — Linear baseline (ridge regression)

Eval notebook. Run `train_baseline.py` first to populate `checkpoints/baseline/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
from pathlib import Path

ROOT = Path("/Users/ewellmeyer/Documents/research/scripts/cloud_feedbacks")
CKPT = ROOT / "checkpoints" / "baseline"

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

y_ga8        = np.load(CKPT / "y_ga8.npy")
y_ga9        = np.load(CKPT / "y_ga9.npy")
y_c2         = np.load(CKPT / "y_c2.npy")
cfmip_fb     = np.load(CKPT / "cfmip_fb.npy")
cfmip_models = np.load(CKPT / "cfmip_models.npy", allow_pickle=True)

y_ga8_cv     = np.load(CKPT / "y_ga8_cv.npy")
y_ga9_pred   = np.load(CKPT / "y_ga9_pred.npy")
y_c2_both    = np.load(CKPT / "y_c2_both_pred.npy")
y_c2_lw      = np.load(CKPT / "y_c2_lw_pred.npy")
y_cfmip_pred = np.load(CKPT / "y_cfmip_pred.npy")
ceres_pred   = float(np.load(CKPT / "ceres_pred.npy"))

def scatter(ax, y_true, y_pred, label, color, marker="o", alpha=0.4, s=8):
    ax.scatter(y_true, y_pred, c=color, alpha=alpha, s=s, marker=marker)
    lo = min(y_true.min(), y_pred.min()) - 0.2
    hi = max(y_true.max(), y_pred.max()) + 0.2
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ax.set_title(f"{label}\nR\u00b2={r2:.3f}  RMSE={rmse:.2f} W/m\u00b2", fontsize=9)
    ax.set_xlabel("Actual \u03b4NetCRE (W/m\u00b2)")
    ax.set_ylabel("Predicted \u03b4NetCRE (W/m\u00b2)")
    return r2, rmse

print("Loaded.")

## Predicted vs actual — GA8 cross-validation

Each point is one ensemble member, predicted from a fold it was not trained on. The 1:1 line is perfect prediction.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
scatter(ax, y_ga8, y_ga8_cv, "GA8 10-fold CV", "steelblue")
plt.tight_layout()
plt.show()

## Out-of-sample: GA9 and CESM2

GA9 tests generalisation across HadGEM physics generations. CESM2 is evaluated twice: both channels (SW is a proxy — expected to be poor) and LW only (fair comparison since GA8 LW was the training signal).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
scatter(axes[0], y_ga9, y_ga9_pred, "GA9 out-of-sample", "darkorange")
scatter(axes[1], y_c2,  y_c2_both,  "CESM2 (both channels, SW proxy)", "tomato")
scatter(axes[2], y_c2,  y_c2_lw,    "CESM2 (LW only)", "seagreen")
plt.suptitle("Out-of-sample — GA8-trained ridge regression", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## CFMIP structural generalisation test

10 models spanning diverse GCM families. Stars mark the PPE parent models (HadGEM3-GC31-LL and CESM2) — they should be the easiest to predict since the training data came from their model families.

In [ ]:
from matplotlib.lines import Line2D

PARENT_MODELS = {"HadGEM3-GC31-LL", "CESM2"}

fig, ax = plt.subplots(figsize=(6, 6))
for mod, yt, yp in zip(cfmip_models, cfmip_fb, y_cfmip_pred):
    is_parent = mod in PARENT_MODELS
    ax.scatter(yt, yp,
               c="crimson" if is_parent else "steelblue",
               marker="*" if is_parent else "o",
               s=120 if is_parent else 60, zorder=3)
    ax.annotate(mod, (yt, yp), fontsize=6.5,
                textcoords="offset points", xytext=(5, 2))

lo = min(cfmip_fb.min(), y_cfmip_pred.min()) - 0.3
hi = max(cfmip_fb.max(), y_cfmip_pred.max()) + 0.3
ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("Actual \u03b4NetCRE (W/m\u00b2)")
ax.set_ylabel("Predicted \u03b4NetCRE (W/m\u00b2)")
ax.set_title("CFMIP structural generalisation (GA8-trained ridge)", fontsize=9)
ax.legend(handles=[
    Line2D([0],[0], marker="*", color="w", markerfacecolor="crimson",  markersize=10, label="PPE parent"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="steelblue", markersize=8,  label="Other CFMIP"),
], fontsize=8)
plt.tight_layout()
plt.show()

## Summary metrics and CERES constraint

All R\u00b2/RMSE numbers in one place, then the observational constraint from applying the model to CERES. This is a linear baseline — treat the CERES number as a reference point.

In [ ]:
rows = [
    ("GA8 10-fold CV",              y_ga8, y_ga8_cv),
    ("GA9 out-of-sample",           y_ga9, y_ga9_pred),
    ("CESM2 (both channels, proxy)", y_c2,  y_c2_both),
    ("CESM2 (LW only)",             y_c2,  y_c2_lw),
    ("CFMIP structural",            cfmip_fb, y_cfmip_pred),
]

print(f"{'Dataset':<35}  {'N':>4}  {'R\u00b2':>7}  {'RMSE (W/m\u00b2)':>12}")
print("-" * 65)
for label, yt, yp in rows:
    r2   = r2_score(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    print(f"{label:<35}  {len(yt):>4d}  {r2:>7.3f}  {rmse:>12.3f}")

print(f"\nCERES observational constraint (linear baseline):")
print(f"  \u03b4NetCRE = {ceres_pred:+.3f} W/m\u00b2")
print(f"  Feedback  = {ceres_pred / 4:+.3f} W/m\u00b2/K")